In [1]:
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 14.9 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system =

In [2]:
!pip install bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 88.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 91.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 59.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 42.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 113.3 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstal

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# 📦 Imports
import os
import json
import torch
import wandb
from datasets import Dataset
import torch.nn.functional as F
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

In [5]:
!wandb login --relogin

wandb: WARNING Using legacy-service, which is deprecated. If this is unintentional, you can fix it by ensuring you do not call `wandb.require('legacy-service')` and do not set the WANDB_X_REQUIRE_LEGACY_SERVICE environment variable.
wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter, or press ctrl+c to quit: 
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


In [6]:
# 🧪 Init W&B
wandb.init(
    project="mental-health-expert-medlineplus-sft",
    name="llama3-8b-ultramedical-mental-health-expert-v1",
    tags=["llama3-8b-ultramedical", "sft", "mental-health", "medical"],
    notes="SFT of llama3-8b-ultramedical for mental-health expertise"
)
#key 3114d04ef3f8187e6f6852dd28ede0fa5a2ec32c

wandb: Currently logged in as: xj2193 (med-moe) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [7]:
# 💾 Save path
model_path = '/content/drive/MyDrive/medmoe/checkpoints/mental_health_bf16_llama3_8b_expert_model'

# 🔧 Hyperparameters
wandb_config = {
    "model_name": "TsinghuaC3I/Llama-3-8B-UltraMedical",
    "learning_rate": 2e-4,
    "epochs": 20,
    "batch_size": 4,
    "gradient_accumulation_steps": 8,
    "lora_r": 16,
    "lora_alpha": 32,
    "medical_domain": "mental-health",
    "load_pretrained": True  # Set to False to load model from scratch
}
wandb.config.update(wandb_config)

In [8]:
# 🧾 Tokenizer
shared_tokenizer_path = '/content/drive/MyDrive/medmoe/shared_tokenizer'
tokenizer = AutoTokenizer.from_pretrained(shared_tokenizer_path)
tokenizer.pad_token = tokenizer.eos_token

In [9]:
# 🧠 Model

model = AutoModelForCausalLM.from_pretrained(
    wandb_config["model_name"],
    device_map="auto",
    torch_dtype=torch.bfloat16
)
model.resize_token_embeddings(len(tokenizer))

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.17G [00:00<?, ?B/s]

...

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/121 [00:00<?, ?B/s]

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
The new lm_head weights will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Embedding(130893, 4096)

In [10]:
# 🧪 LoRA Config
lora_config = LoraConfig(
    r=wandb_config["lora_r"],
    lora_alpha=wandb_config["lora_alpha"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)
model = get_peft_model(model, lora_config)

In [11]:
# 📂 Load Dataset
with open("/content/drive/MyDrive/medmoe/mental_behavior_qa.json", "r") as f:
    qa_data = json.load(f)

train_data = []
for topic in qa_data:
    for question, answer in topic['question_answer_pair']:
        prompt = "Answer this question about mental health: "
        train_data.append({"text": prompt + question,
                           "reference": answer
                           })

dataset = Dataset.from_list(train_data).train_test_split(test_size=0.1)

In [12]:
# 🔁 Tokenize
def tokenize(example):
    model_inputs = tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=512
    )

    labels = tokenizer(
        example["reference"],
        padding="max_length",
        truncation=True,
        max_length=512
    )

    model_inputs["labels"] = labels['input_ids']
    return model_inputs

tokenized = dataset.map(
    tokenize,
    batched=True,
    remove_columns=dataset["train"].column_names
)

Map:   0%|          | 0/129 [00:00<?, ? examples/s]

Map:   0%|          | 0/15 [00:00<?, ? examples/s]

In [13]:
from peft import PeftModel
from typing import Optional
import os
import torch
from collections import OrderedDict
from safetensors.torch import save_file

class SemanticTrainer(Trainer):
    def __init__(self, tokenizer, *args, **kwargs):
        kwargs["processing_class"] = tokenizer
        super().__init__(*args, **kwargs)
        self.tokenizer = tokenizer
        self._signature_columns = ['input_ids', 'attention_mask', 'labels']

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        input_ids = inputs.get("input_ids")
        attention_mask = inputs.get("attention_mask")
        labels = inputs.get("labels")

        input_ids = input_ids.to(model.device)
        attention_mask = attention_mask.to(model.device) if attention_mask is not None else None
        labels = labels.to(model.device) if labels is not None else None

        # Generate answers
        outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        output_hidden_states=True,
        return_dict=True
        )

        # Decode generated text
        hidden_states = outputs.hidden_states[-1]

        # Mean pooling over non-padding tokens
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(hidden_states.size())
        sum_embeddings = torch.sum(hidden_states * input_mask_expanded, dim=1)
        sum_mask = input_mask_expanded.sum(dim=1)
        gen_embeddings = sum_embeddings / torch.clamp(sum_mask, min=1e-9)

        with torch.no_grad():
            ref_attention_mask = (labels != self.tokenizer.pad_token_id).to(model.device)
            ref_outputs = model(
                input_ids=labels,
                attention_mask=ref_attention_mask,
                output_hidden_states=True,
                return_dict=True
            )
            ref_hidden = ref_outputs.hidden_states[-1]
            ref_mask_expanded = ref_attention_mask.unsqueeze(-1).expand(ref_hidden.size())
            sum_ref_embeds = torch.sum(ref_hidden * ref_mask_expanded, dim=1)
            sum_ref_mask = ref_mask_expanded.sum(dim=1)
            ref_embeddings = sum_ref_embeds / torch.clamp(sum_ref_mask, min=1e-9)

        # Compute cosine similarity as reward
        sim = F.cosine_similarity(gen_embeddings, ref_embeddings, dim=-1)
        loss = 1 - sim.clamp(min=-1.0, max=1.0).mean()

        return (loss, outputs) if return_outputs else loss


    def save_model(self, output_dir: Optional[str] = None, _internal_call: bool = False):
        output_dir = output_dir if output_dir is not None else self.args.output_dir
        os.makedirs(output_dir, exist_ok=True)

        print(f"Type of self.model during saving: {type(self.model)}")
        print("Is self.model a PeftModel?", isinstance(self.model, PeftModel))

        # Ensure you're saving from the actual PeftModel
        self.model.save_pretrained(output_dir, save_adapter=True)

        self.tokenizer.save_pretrained(output_dir)

        # Show size for debugging
        adapter_path = os.path.join(output_dir, "adapter_model.safetensors")
        if os.path.exists(adapter_path):
            file_size = os.path.getsize(adapter_path) / (1024 * 1024)
            print(f"Actual saved adapter size: {file_size:.2f} MB")
        else:
            print("adapter_model.safetensors not found.")


In [14]:
# ⚙️ Training Args
training_args = TrainingArguments(
    output_dir=model_path,
    per_device_train_batch_size=wandb_config["batch_size"],
    per_device_eval_batch_size=wandb_config["batch_size"],
    gradient_accumulation_steps=wandb_config["gradient_accumulation_steps"],
    num_train_epochs=wandb_config["epochs"],
    learning_rate=wandb_config["learning_rate"],
    remove_unused_columns=False,
    logging_dir="./logs",
    logging_steps=10,
    save_strategy="steps",
    save_steps=10,

    fp16=False,
    bf16=True,
    save_total_limit=5,
    report_to="wandb",
    metric_for_best_model="loss",
    save_safetensors=True
)

trainer = SemanticTrainer(
    model=model,
    tokenizer=tokenizer,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"]
)

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
Trainer.tokenizer is now deprecated. You should use `Trainer.processing_class = processing_class` instead.


In [15]:
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1' # Set this before trainer.train() call

In [16]:
print(type(model))

<class 'peft.peft_model.PeftModelForCausalLM'>


In [17]:
# 🚀 Train
trainer.train()
trainer.save_model('/content/drive/MyDrive/medmoe/model/mental_health_bf16_llama3_8b_expert_model')
print("Training completed and model saved successfully!")

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.


Step,Training Loss
10,1.377200
20,0.102200
30,0.042800
40,0.013500
50,0.008400
60,0.007400
70,0.007100
80,0.006900


Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Tr

Type of self.model during saving: <class 'peft.peft_model.PeftModelForCausalLM'>
Is self.model a PeftModel? True


/usr/local/lib/python3.11/dist-packages/peft/utils/save_and_load.py:250: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.


...

Type of self.model during saving: <class 'peft.peft_model.PeftModelForCausalLM'>
Is self.model a PeftModel? True


/usr/local/lib/python3.11/dist-packages/peft/utils/save_and_load.py:250: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.


Actual saved adapter size: 2205.26 MB
Type of self.model during saving: <class 'peft.peft_model.PeftModelForCausalLM'>
Is self.model a PeftModel? True


/usr/local/lib/python3.11/dist-packages/peft/utils/save_and_load.py:250: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.


Actual saved adapter size: 2205.26 MB
Training completed and model saved successfully!
